<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/PreferredAI/tutorials/blob/master/recommender-systems/11_next_item_recommendation.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/PreferredAI/tutorials/blob/master/recommender-systems/11_next_item_recommendation.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# Session-based Recommendation

Many real-world recommendation settings have **no long-term user profile**: an
anonymous visitor lands on a site, clicks a few items, and we must predict what
they will interact with *next*, all from the current session alone. This is the
**session-based next-item** task.

This tutorial **builds the model family up one idea at a time**, and at every step the new model has to earn its place against the previous one:

1. **SPop**: how far does plain *popularity* get us, with no learning at all? (Section 3)
2. **FPMC**: add the smallest bit of order: a first-order *Markov chain* over the last click. (Section 4)
3. **GRU4Rec**: encode the *whole* session with a recurrent net, for *long-term* dependencies. (Section 5)
4. **SASRec / BERT4Rec**: replace recurrence with *self-attention*, causal and bidirectional. (Section 6)
5. **Loss functions**: is the ranking *loss* worth tuning? (Sections 7)

Cornac ships all of these behind the same `NextItemRecommender` interface and the same training substrate (a session iterator, a shared set of ranking losses, and best-on-validation model selection), so as we go down the list the only things that change are the **encoder** and the **loss**:

| Model | Core idea | Encoder | Year |
|---|---|---|---|
| **SPop** | in-session + global popularity | counts (no learning) | - |
| **FPMC** | first-order Markov chain (last item) | factorization | 2010 |
| **GRU4Rec** | full history, recurrent | GRU | 2015 |
| **SASRec** | full history, causal attention | Transformer (causal) | 2018 |
| **BERT4Rec** | full history, bidirectional attention | Transformer (BERT) | 2019 |

We train them on the **Diginetica** dataset, watching each model move the numbers, and close with a pre-computed sweep over the loss function.

## 1. Setup

In [1]:
!pip install --quiet cornac==2.5.0

In [2]:
import sys

import numpy as np
import pandas as pd
import torch

import cornac
from cornac.datasets import diginetica
from cornac.eval_methods import NextItemEvaluation
from cornac.metrics import MRR, NDCG, Recall
from cornac.models import BERT4Rec, FPMC, GRU4Rec, SASRec, SPop
from cornac.utils import cache

SEED = 123
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"System version : {sys.version.split()[0]}")
print(f"Cornac version : {cornac.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device         : {DEVICE}")

System version : 3.12.13
Cornac version : 2.5.0
PyTorch version: 2.11.0+cu128
Device         : cuda:0


## 2. The Diginetica dataset

`cornac.datasets.diginetica` is an e-commerce click logs session-based dataset used in the
[CIKM Cup 2016 Diginetica](https://competitions.codalab.org/forums/7901/1941/).

In [3]:
train_data = diginetica.load_train()
val_data = diginetica.load_val()
test_data = diginetica.load_test()

next_item_eval = NextItemEvaluation.from_splits(
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    exclude_unknowns=True,
    verbose=True,
    fmt="USIT",
)

rating_threshold = 1.0
exclude_unknowns = True
---
Training data:
Number of users = 571
Number of items = 4199
Number of sessions = 1528
---
Test data:
Number of users = 571
Number of items = 4199
Number of sessions = 416
Number of unknown users = 0
Number of unknown items = 0
---
Validation data:
Number of users = 571
Number of items = 4199
Number of sessions = 451
---
Total users = 571
Total items = 4199
Total sessions = 2395


## 3. A popularity baseline: SPop

If we were starting from scratch, the simplest thing we could do is recommend whatever is **popular**, no sequence modelling at all. Cornac's `SPop` is the session-based *popularity* baseline, summing two signals:

- **Session popularity**: items the visitor has already clicked *in the current session* (integer counts, the dominant term over global popularity).
- **Global popularity**: item frequency over the whole training set (normalized to `[0, 1]`), used to break ties and to fill the ranking when we need more items than the short session has seen (e.g. `Recall@50` on a 5-click session).

Why is this not a silly baseline? Because **session behaviour repeats**.
Picture a shopper who already has a product in mind: they land on the site, search for it, then bounce between it and a couple of competitors before deciding, a click stream like

```
A → B → A → C → A
```

The item you just saw is very often the item you will see again. That makes "what's already popular in this session" a genuinely strong place to start, and the floor every learned model has to beat.

Let's try `SPop` with the above example:`[A, B, A, C] → A`, with the current session as `[A, B, A, C]` and the target is `A`.
- Session popularity: `A: 2, B: 1, C: 1`
- Global popularity: assume in the global training set, we have `A: 0.1, B: 0.2, C:0.3, D:0.4, E:0.5`

So the predicted scores would be: `A: 2.1, B: 1.2, C: 1.3, D:0.4, E:0.5`.

The recommended item order would be `[A, C, B, E, D]`.

In [4]:
metrics = [MRR(), NDCG(k=10), NDCG(k=50), Recall(k=10), Recall(k=50)]

spop = SPop()

cornac.Experiment(
    eval_method=next_item_eval,
    models=[spop],
    metrics=metrics,
).run()


[SPop] Training started!

[SPop] Evaluation started!


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


VALIDATION:
...
     |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Time (s)
---- + ------ + ------- + ------- + --------- + --------- + --------
SPop | 0.2376 |  0.2490 |  0.2590 |    0.2926 |    0.3376 |   0.6626

TEST:
...
     |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Train (s) | Test (s)
---- + ------ + ------- + ------- + --------- + --------- + --------- + --------
SPop | 0.2344 |  0.2432 |  0.2506 |    0.2776 |    0.3110 |    0.0019 |   0.5759



Even with no training, SPop is respectable: the held-out next item lands in the top-10 more than a **fourth** of the time (`Recall@10 ≈ 0.28`), with `NDCG@10 ≈ 0.24`. Hold on to these numbers, they are the bar every model below has to clear.

## 4. Adding short-term order: FPMC

`SPop` ignores **order** entirely, it only counts. The step up is to model a single transition: *given the item you just clicked, what comes next?*

`FPMC` (Factorizing Personalized Markov Chains) does this with a **first-order Markov chain**, personalized by a user factor and **factorized** into embeddings to predict the (last-item → next-item) sequence. Because it models a transition rather than raw frequency, it *oughts to* improve on SPop.

In [5]:
fpmc = FPMC(
    embedding_dim=100,
    loss="cross-entropy",
    n_sample=512,
    batch_size=128,
    learning_rate=0.1,
    n_epochs=100,
    model_selection="best",
    val_eval_every=5,
    val_metric="ndcg",
    val_k=10,
    device=DEVICE,
    verbose=True,
    seed=SEED,
)

cornac.Experiment(
    eval_method=next_item_eval,
    models=[spop, fpmc],
    metrics=metrics,
).run()


[SPop] Training started!

[SPop] Evaluation started!


/usr/local/lib/python3.12/dist-packages/cornac/models/recommender.py:322: UserWarning: Model is already fitted. Re-fitting will overwrite the previous model.
  warnings.warn(


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


[FPMC] Training started!


  0%|          | 0/100 [00:00<?, ?it/s]


[FPMC] Evaluation started!


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


VALIDATION:
...
     |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Time (s)
---- + ------ + ------- + ------- + --------- + --------- + --------
SPop | 0.2376 |  0.2490 |  0.2590 |    0.2926 |    0.3376 |   0.4865
FPMC | 0.1860 |  0.2146 |  0.2337 |    0.3248 |    0.4084 |   0.4812

TEST:
...
     |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Train (s) | Test (s)
---- + ------ + ------- + ------- + --------- + --------- + --------- + --------
SPop | 0.2344 |  0.2432 |  0.2506 |    0.2776 |    0.3110 |    0.0022 |   0.4524
FPMC | 0.1701 |  0.1993 |  0.2155 |    0.3077 |    0.3746 |   29.3559 |   0.4615



FPMC behaves more subtly than "strictly better". Its **Recall** rises above SPop, modelling the last transition really does retrieve the right item more often, but its **NDCG** slips slightly *below* SPop. In other words, it identifies the target correctly more often, yet ranks it a little worse. So, a single, personalized last-item hop is informative, but on its own not enough to out-rank a strong popularity prior.

## 5. Long-term dependencies: GRU4Rec

A first-order chain has a one-item memory: it forgets everything before the last click.

`GRU4Rec` instead runs a **GRU** over the *entire* session, folding the whole prefix into a hidden state and predicting from it. That lets it capture **long-term** dependencies, an item three or four clicks back can still steer the next prediction, which `FPMC` structurally cannot do.

In [6]:
gru4rec = GRU4Rec(
    layers=[100],
    loss="cross-entropy",
    dropout_p_hidden=0.3,
    sample_alpha=0.75,
    n_sample=512,
    batch_size=64,
    learning_rate=0.1,
    n_epochs=100,
    model_selection="best",
    val_eval_every=5,
    val_metric="recall",
    val_k=20,
    device=DEVICE,
    verbose=True,
    seed=SEED,
)

cornac.Experiment(
    eval_method=next_item_eval,
    models=[spop, fpmc, gru4rec],
    metrics=metrics,
).run()


[SPop] Training started!

[SPop] Evaluation started!


/usr/local/lib/python3.12/dist-packages/cornac/models/recommender.py:322: UserWarning: Model is already fitted. Re-fitting will overwrite the previous model.
  warnings.warn(


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


[FPMC] Training started!


...

  0%|          | 0/100 [00:00<?, ?it/s]


[GRU4Rec] Evaluation started!


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


VALIDATION:
...
        |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Time (s)
------- + ------ + ------- + ------- + --------- + --------- + --------
SPop    | 0.2376 |  0.2490 |  0.2590 |    0.2926 |    0.3376 |   0.4903
FPMC    | 0.1826 |  0.2133 |  0.2313 |    0.3280 |    0.4084 |   0.4655
GRU4Rec | 0.3252 |  0.3623 |  0.3768 |    0.4887 |    0.5531 |   0.7804

TEST:
...
        |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Train (s) | Test (s)
------- + ------ + ------- + ------- + --------- + --------- + --------- + --------
SPop    | 0.2344 |  0.2432 |  0.2506 |    0.2776 |    0.3110 |    0.0013 |   0.4639
FPMC    | 0.1757 |  0.2040 |  0.2197 |    0.3077 |    0.3746 |   23.9825 |   0.4713
GRU4Rec | 0.2901 |  0.3275 |  0.3446 |    0.4582 |    0.5351 |   64.5778 |   0.6799



Reading the whole prefix changes the picture decisively: GRU4Rec beats **both** earlier models on **every** metric, and not by a little. `NDCG@10` jumps from the ~0.21-0.24 range up to ~0.33. This is the payoff of long-term dependencies: context from early in the session, not just the last click, now shapes the prediction.

## 6. Attention instead of recurrence: SASRec & BERT4Rec

A GRU squeezes the entire history through a single hidden state passed step by step, so signal from early clicks must survive many updates (the classic **vanishing-gradient** problem). **Self-attention** removes the recurrence: every position attends *directly* to every other, so a click ten steps back is only one hop away.

### Same recipe: many-to-one, last hidden state only

All the sequence models in Cornac (GRU4Rec, SASRec **and** BERT4Rec) are trained the same **causal-LM, many-to-one** way. The session iterator slices each session into growing prefixes, each paired with the click that follows it. For a session `[a, b, c, d] → e`:

| input (history) | target (next click) |
|---|---|
| `[a]` | `b` |
| `[a, b]` | `c` |
| `[a, b, c]` | `d` |
| `[a, b, c, d]` | `e` |

So one session of length *L* becomes *L − 1* separate (prefix → next-item) examples. For each one, the encoder reads the prefix and we score the next item from the **last position's hidden state only**, not an average over positions and not a per-token output.

In [7]:
transformer = dict(
    learning_rate=0.01,
    embedding_dim=100,
    loss="cross-entropy",
    n_sample=512,
    batch_size=128,
    n_epochs=100,
    max_len=20,
    num_blocks=2,
    num_heads=2,
    model_selection="best",
    val_eval_every=5,
    val_metric="ndcg",
    val_k=10,
    device=DEVICE,
    verbose=True,
    seed=SEED,
)

sasrec = SASRec(**transformer)
bert4rec = BERT4Rec(**transformer)

cornac.Experiment(
    eval_method=next_item_eval,
    models=[spop, fpmc, gru4rec, sasrec, bert4rec],
    metrics=metrics,
).run()


[SPop] Training started!

[SPop] Evaluation started!


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


[FPMC] Training started!


  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]


[BERT4Rec] Evaluation started!


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


VALIDATION:
...
         |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Time (s)
-------- + ------ + ------- + ------- + --------- + --------- + --------
SPop     | 0.2376 |  0.2490 |  0.2590 |    0.2926 |    0.3376 |   0.4831
FPMC     | 0.1851 |  0.2143 |  0.2325 |    0.3248 |    0.4051 |   0.4960
GRU4Rec  | 0.3196 |  0.3556 |  0.3708 |    0.4791 |    0.5466 |   0.5579
SASRec   | 0.3245 |  0.3545 |  0.3711 |    0.4598 |    0.5370 |   1.4015
BERT4Rec | 0.3388 |  0.3696 |  0.3795 |    0.4727 |    0.5177 |   1.4084

TEST:
...
         |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Train (s) | Test (s)
-------- + ------ + ------- + ------- + --------- + --------- + --------- + --------
SPop     | 0.2344 |  0.2432 |  0.2506 |    0.2776 |    0.3110 |    0.0024 |   0.4785
FPMC     | 0.1772 |  0.2045 |  0.2203 |    0.3043 |    0.3746 |   24.5325 |   0.4520
GRU4Rec  | 0.3049 |  0.3427 |  0.3568 |    0.4716 |    0.5351 |   56.8069 |   0.5736
SASRec   | 0.3161 |  0.3505 |  0.366

Both attention encoders land in the same top tier as GRU4Rec, on par with each other (on NDCG, as we choose the best model on val_NDCG@10). The progress we have made is now clear:

> **popularity  <  single transition  <  full-history recurrence  ≲  full-history attention**

Each step bought a real improvement by letting the model condition on *more* of the session.

A caveat on this dataset, though: Diginetica sessions are **short**, averaging only ~4.55 interactions per session. With so little history to attend over, SASRec and BERT4Rec cannot really stretch their long-range capacity, which is part of why they only edge past GRU4Rec here. On denser datasets with longer sessions there is more context to exploit, and the gap of attention over recurrence (and the room for the two attention models to diverge) is expected to widen.

## 7. Does the loss function matter?

So far, every model above was trained with `cross-entropy`.

In `Cornac`, we implement various loss functions, including:

- **`cross-entropy`**, softmax over all columns, maximize the diagonal (a.k.a. `xe_softmax`; `ce`).
- **`bpr`**, Bayesian Personalized Ranking; pairwise log-sigmoid of *(positive - negative)*.
- **`bpr-max`**, BPR-max (Hidasi & Karatzoglou, 2018): softmax-weighted negatives with a score regularizer.
- **`top1`**, the TOP1 ranking loss from the original GRU4Rec paper.
- **`bce`**, binary cross-entropy treating the diagonal as 1 and every other column as 0.

Swapping the loss is a one-argument change, e.g. `SASRec(loss="bpr", ...)`.

Running the full grid live would take too long, so we load a **pre-computed sweep** (`tuning_results.csv`): each of the **four learned models** (SPop has nothing to tune) trained with every loss at several learning rates.

We follow the usual protocol to avoid fooling ourselves:

1. **Select on validation.** For each (model, loss) we keep the learning rate with the best `val_NDCG@10`. The test set is never looked at while choosing hyperparameters.
2. **Report the private (test) leaderboard.** We then read off how those val-selected configs do on the held-out **test** set, and rank by `val_NDCG@10`. This "private leaderboard" is the honest estimate of generalization, analogous to a competition's hidden test split.

In [8]:
cache("http://static.preferred.ai/cornac/datasets/diginetica/tuning_results.csv", unzip=False)

'/root/.cornac/tuning_results.csv'

In [9]:
CSV_PATH = "/root/.cornac/tuning_results.csv"

results = pd.read_csv(CSV_PATH)
results = results[results["model"] != "GPT2Rec"]  # not covered in this tutorial

# 1) Best val NDCG@10 for each (model, loss)
best = results.sort_values("val_NDCG@10", ascending=False).groupby(["model", "loss"], as_index=False).first()

# 2) Private test with those val-selected configs
leaderboard = best.sort_values("val_NDCG@10", ascending=False).reset_index(drop=True)
leaderboard[
    [
        "model",
        "loss",
        "learning_rate",
        "val_NDCG@10",
        "test_NDCG@10",
        "test_NDCG@50",
        "test_Recall@10",
        "test_Recall@50",
        "test_MRR",
    ]
]

,model,loss,learning_rate,val_NDCG@10,test_NDCG@10,test_NDCG@50,test_Recall@10,test_Recall@50,test_MRR
0,BERT4Rec,bpr,0.0100,0.3916,0.3704,0.3814,0.4749,0.5251,0.3402
1,SASRec,bpr,0.0100,0.3913,0.3723,0.3861,0.4716,0.5351,0.3435
2,BERT4Rec,bpr-max,0.0010,0.3897,0.3682,0.3798,0.4482,0.5017,0.3454
3,SASRec,bce,0.0005,0.3888,0.3746,0.3894,0.4883,0.5552,0.3419
4,SASRec,bpr-max,0.0005,0.3851,0.3688,0.3779,0.4649,0.5050,0.3408
5,BERT4Rec,top1,0.0010,0.3837,0.3555,0.3632,0.4482,0.4816,0.3281
6,SASRec,cross-entropy,0.0030,0.3837,0.3739,0.3864,0.4783,0.5351,0.3441
7,BERT4Rec,bce,0.0030,0.3822,0.3483,0.3675,0.4749,0.5585,0.3126
8,SASRec,top1,0.0100,0.3787,0.3619,0.3783,0.4983,0.5719,0.3222
9,BERT4Rec,cross-entropy,0.0030,0.3747,0.3607,0.3747,0.4716,0.5318,0.3292


We choose the "best" models by val_NDCG@10, but that doesn't guarantee the "best" performance on test set.

In [10]:
# Model x loss heatmap of test NDCG@10 (higher = better).
pivot = best.pivot(index="model", columns="loss", values="test_NDCG@10")
model_order = pivot.max(axis=1).sort_values(ascending=False).index
loss_order = pivot.max(axis=0).sort_values(ascending=False).index
pivot = pivot.loc[model_order, loss_order]
pivot.style.background_gradient(cmap="Greens", axis=None).format("{:.4f}")

loss,bce,cross-entropy,bpr,bpr-max,top1
model,,,,,
SASRec,0.3746,0.3739,0.3723,0.3688,0.3619
BERT4Rec,0.3483,0.3607,0.3704,0.3682,0.3555
GRU4Rec,0.0146,0.3510,0.3156,0.1501,0.3482
FPMC,0.0177,0.2134,0.1736,0.2000,0.2216


Read this table two ways: **down a column** to see which encoder suits a given loss, and **across a row** to see how sensitive a model is to its loss. The striking pattern is **robustness**: the Transformer encoders (SASRec, BERT4Rec) score within a narrow band across every loss, whereas `FPMC` and `GRU4Rec` *collapse* under pointwise `bce` and need a ranking loss (`cross-entropy`, `bpr`, `top1`) to train at all.

## References

1. Hidasi, B., Karatzoglou, A., Baltrunas, L., & Tikk, D. (2015). *Session-based
   Recommendations with Recurrent Neural Networks.* arXiv:1511.06939. (GRU4Rec; also the
   `SPop` baseline.)
2. Rendle, S., Freudenthaler, C., & Schmidt-Thieme, L. (2010). *Factorizing
   Personalized Markov Chains for Next-Basket Recommendation.* WWW. (FPMC)
3. Kang, W.-C., & McAuley, J. (2018). *Self-Attentive Sequential Recommendation.*
   ICDM. arXiv:1808.09781. (SASRec)
4. Sun, F., et al. (2019). *BERT4Rec: Sequential Recommendation with Bidirectional
   Encoder Representations from Transformer.* CIKM. arXiv:1904.06690.
5. Hidasi, B., & Karatzoglou, A. (2018). *Recurrent Neural Networks with Top-k Gains
   for Session-based Recommendations.* CIKM. (BPR-max / TOP1 losses)